# 03 — Inference with the saved model

Loads `models/model.joblib` (preprocessing + classifier in one pipeline) and scores raw patient records — the same code path used by the API.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import json
import pandas as pd

from src.data_processing import TARGET, load_clean
from src.predict import SAMPLE_PATIENT, load_metadata, load_model, predict

## 1. Model metadata

In [2]:
meta = load_metadata()
print("Model:", meta["model_name"], "| trained:", meta["trained_at"])
print("Selected by:", meta["selection_metric"])
pd.Series(meta["test_metrics"]).round(3)

Model: logistic_regression | trained: 2026-09-30T11:57:21+00:00
Selected by: recall


accuracy     0.867
precision    0.885
recall       0.821
f1           0.852
roc_auc      0.938
dtype: float64

## 2. Score a single patient (raw JSON-style input)

In [3]:
print(json.dumps(SAMPLE_PATIENT, indent=2))
predict(SAMPLE_PATIENT)

{
  "age": 54,
  "sex": 1,
  "cp": 4,
  "trestbps": 130,
  "chol": 250,
  "fbs": 0,
  "restecg": 2,
  "thalach": 140,
  "exang": 1,
  "oldpeak": 1.6,
  "slope": 2,
  "ca": 1,
  "thal": 7
}


[{'prediction': 1,
  'label': 'disease',
  'confidence': 0.9337,
  'probability_disease': 0.9337,
  'risk_level': 'high'}]

## 3. Batch scoring and comparison with true labels

In [4]:
df = load_clean().sample(8, random_state=7)
preds = pd.DataFrame(predict(df.drop(columns=TARGET).to_dict(orient="records")))
preds.insert(0, "actual", df[TARGET].values)
preds

,actual,prediction,label,confidence,probability_disease,risk_level
0,0,0,no disease,0.6376,0.3624,moderate
1,0,0,no disease,0.7620,0.2380,low
2,0,0,no disease,0.7411,0.2589,low
3,0,0,no disease,0.8760,0.1240,low
4,1,1,disease,0.9793,0.9793,high
5,1,1,disease,0.6087,0.6087,moderate
6,0,0,no disease,0.8000,0.2000,low
7,0,0,no disease,0.8080,0.1920,low


## 4. Reloading via MLflow (registered model)

The same model is also registered in MLflow as `heart-disease-classifier`.

In [5]:
import mlflow

mlflow.set_tracking_uri(f"sqlite:///{(ROOT / 'mlflow.db').as_posix()}")
try:
    mlflow_model = mlflow.sklearn.load_model("models:/heart-disease-classifier/latest")
    print("MLflow model loaded:", type(mlflow_model).__name__)
    print(predict(SAMPLE_PATIENT, model=mlflow_model))
except Exception as exc:  # registry only exists after running src.train locally
    print("MLflow registry not available:", exc)

2026/10/01 13:09:25 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at C:\Users\GU\third-sem\MLOps\assignment1\.venv\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


MLflow model loaded: Pipeline
[{'prediction': 1, 'label': 'disease', 'confidence': 0.9337, 'probability_disease': 0.9337, 'risk_level': 'high'}]
